# Notebook 14: Planning & Memory for Agents

**Series**: Frontier ML Interview Prep — Agentic Systems  
**Prerequisites**: Notebook 13 (Agent Foundations: ReAct & Tool Use)  
**Goal**: Build planning and memory systems from scratch. No frameworks — understand every component.

Notebook 13 built a reactive agent (observe → think → act). But for complex tasks, reactive behavior isn't enough. You need:
- **Planning**: decompose complex goals into manageable subtasks
- **Memory**: remember past observations, experiences, and lessons

---

## 0. Self-Quiz (Active Recall — attempt BEFORE reading)

Write your answers in the cell below before proceeding. Return here after completing the notebook.

1. **What are the 3 types of agent memory?** How do they differ?
2. **How does task decomposition work?** Why is it necessary?
3. **What is Tree of Thoughts?** How does it differ from chain-of-thought?
4. **When does planning fail?** What are the failure modes?
5. **What is the difference between episodic and semantic memory in agents?**

In [ ]:
# YOUR ANSWERS HERE (fill in before reading the notebook)
self_quiz_answers = {
    "three_types_of_memory": "",
    "task_decomposition": "",
    "tree_of_thoughts": "",
    "when_planning_fails": "",
    "episodic_vs_semantic": "",
}

---
## 1. Setup

In [ ]:
!pip install -q openai faiss-cpu numpy torch transformers sentence-transformers

In [ ]:
import os
import re
import json
import time
import math
import hashlib
import traceback
import numpy as np
from typing import Any, Dict, List, Optional, Callable, Tuple
from dataclasses import dataclass, field
from datetime import datetime
from abc import ABC, abstractmethod
from collections import defaultdict
import textwrap

# For Colab: set your API key
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Or set directly:
# os.environ["OPENAI_API_KEY"] = "sk-..."

from openai import OpenAI

USE_MOCK_LLM = not os.environ.get("OPENAI_API_KEY")
if USE_MOCK_LLM:
    print("No OPENAI_API_KEY found. Running with mock LLM for demonstration.")
    print("Set your API key to use real LLM calls.")
else:
    client = OpenAI()
    print("OpenAI client initialized.")

---
## 2. Reuse: Tool & Agent Infrastructure from Notebook 13

We redefine the core building blocks from Notebook 13 so this notebook is self-contained.

In [ ]:
# ---- Tool abstraction (from Notebook 13) ----

class Tool(ABC):
    """Abstract base class for agent tools."""
    @property
    @abstractmethod
    def name(self) -> str: pass
    
    @property
    @abstractmethod
    def description(self) -> str: pass
    
    @property
    @abstractmethod
    def parameters(self) -> Dict[str, Any]: pass
    
    @abstractmethod
    def execute(self, **kwargs) -> str: pass
    
    def format_for_prompt(self) -> str:
        params_desc = []
        for pname, pinfo in self.parameters.get("properties", {}).items():
            req = "(required)" if pname in self.parameters.get("required", []) else "(optional)"
            params_desc.append(f"    - {pname} ({pinfo.get('type','string')}): {pinfo.get('description','')} {req}")
        params_str = "\n".join(params_desc) if params_desc else "    (no parameters)"
        return f"Tool: {self.name}\nDescription: {self.description}\nParameters:\n{params_str}"


class CalculatorTool(Tool):
    @property
    def name(self): return "Calculator"
    @property
    def description(self): return "Evaluates a mathematical expression. Supports arithmetic, math functions (sqrt, log, etc.), and constants (pi, e)."
    @property
    def parameters(self):
        return {"type": "object", "properties": {"expression": {"type": "string", "description": "Math expression to evaluate"}}, "required": ["expression"]}
    def execute(self, expression="", **kw):
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        allowed.update({'abs': abs, 'round': round, 'min': min, 'max': max, 'int': int, 'float': float})
        try:
            return str(eval(expression, {"__builtins__": {}}, allowed))
        except Exception as e:
            return f"Error: {e}"


class SearchTool(Tool):
    def __init__(self):
        self._kb = {
            "RLHF": "Reinforcement Learning from Human Feedback (RLHF) fine-tunes LLMs using human preference data. Key papers: InstructGPT (Ouyang et al. 2022), which used a 3-step process: SFT, reward model training, PPO. Constitutional AI (Bai et al. 2022) replaces humans with AI feedback. DPO (Rafailov et al. 2023) simplifies RLHF by eliminating the reward model.",
            "DPO": "Direct Preference Optimization (DPO) by Rafailov et al. (2023) simplifies RLHF by directly optimizing the policy using a classification loss on preference pairs, avoiding the need for a separate reward model. The key insight: the optimal policy under a KL-constrained reward maximization has a closed-form relationship to the reward function.",
            "transformer": "The Transformer (Vaswani et al. 2017) uses self-attention for parallel sequence processing. Core: multi-head attention, positional encoding, layer norm, FFN. Scaled dot-product attention: softmax(QK^T/sqrt(d_k))V.",
            "attention": "Attention computes weighted sum of values based on query-key similarity. Multi-head attention runs h parallel attention functions. Flash Attention (Dao et al. 2022) makes attention memory-efficient using tiling.",
            "ReAct": "ReAct (Yao et al. 2022) interleaves reasoning (chain-of-thought) with actions (tool use). Format: Thought/Action/Observation loop. Outperforms CoT-only and Act-only on knowledge-intensive and decision-making tasks.",
            "tree of thoughts": "Tree of Thoughts (Yao et al. 2023) extends CoT by exploring multiple reasoning paths. Uses BFS/DFS search over thought sequences with LLM-based evaluation. Enables backtracking when a path fails.",
            "multi-agent": "Multi-agent systems use multiple LLM agents for collaboration. Approaches: debate (agents argue), division of labor (specialist agents), hierarchical (manager delegates). Challenges: coordination overhead, error propagation.",
            "toolformer": "Toolformer (Schick et al. 2023) trains LLMs to decide when and how to use tools (calculator, search, calendar, etc.) by self-supervised learning. The model learns to insert API calls into text where they would be helpful.",
            "memory agents": "Agent memory systems: Short-term (context window), Long-term (vector store for retrieval), Episodic (past task experiences). Working memory management: summarize old context to stay within token limits.",
            "planning AI": "AI planning approaches: Task decomposition (break complex goals into subtasks), Hierarchical planning (high-level then low-level), Re-planning (adapt when subtasks fail). Key challenge: plans often don't survive contact with reality.",
            "Python": "Python is a high-level programming language. Version 3.12+ supports performance improvements. Popular for ML/AI.",
        }
    @property
    def name(self): return "Search"
    @property
    def description(self): return "Searches the web for information. Returns relevant results as text."
    @property
    def parameters(self):
        return {"type": "object", "properties": {"query": {"type": "string", "description": "Search query"}}, "required": ["query"]}
    def execute(self, query="", **kw):
        q = query.lower()
        results = [v for k, v in self._kb.items() if any(w in q for w in k.lower().split())]
        return "\n\n".join(results[:3]) if results else f"No results for '{query}'."


class PythonREPLTool(Tool):
    def __init__(self):
        self._ns = {"__builtins__": {"print": print, "len": len, "range": range, "enumerate": enumerate, "zip": zip, "map": map, "filter": filter, "sorted": sorted, "list": list, "dict": dict, "set": set, "tuple": tuple, "str": str, "int": int, "float": float, "bool": bool, "sum": sum, "min": min, "max": max, "abs": abs, "round": round, "isinstance": isinstance, "type": type, "Exception": Exception, "True": True, "False": False, "None": None}, "math": math, "json": json}
    @property
    def name(self): return "PythonREPL"
    @property
    def description(self): return "Executes Python code and returns output. Use print() for output."
    @property
    def parameters(self):
        return {"type": "object", "properties": {"code": {"type": "string", "description": "Python code to execute"}}, "required": ["code"]}
    def execute(self, code="", **kw):
        import io, sys
        old = sys.stdout; sys.stdout = cap = io.StringIO()
        try:
            exec(code, self._ns)
            out = cap.getvalue()
            return out.strip() if out.strip() else "(executed, no output)"
        except Exception as e:
            return f"Error: {type(e).__name__}: {e}"
        finally:
            sys.stdout = old


class ToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}
    def register(self, tool: Tool): self._tools[tool.name] = tool
    def get(self, name: str) -> Optional[Tool]:
        if name in self._tools: return self._tools[name]
        for n, t in self._tools.items():
            if n.lower() == name.lower(): return t
        return None
    def list_tools(self) -> List[str]: return list(self._tools.keys())
    def format_for_prompt(self) -> str:
        return "\n\n".join(t.format_for_prompt() for t in self._tools.values())

print("Tool infrastructure loaded.")

In [ ]:
# ---- Minimal ReAct Agent (from Notebook 13, simplified) ----

class SimpleReActAgent:
    """Minimal ReAct agent for use as a subtask executor."""
    
    def __init__(self, tools: Optional[List[Tool]] = None, max_steps: int = 5, verbose: bool = False):
        self.max_steps = max_steps
        self.verbose = verbose
        self.registry = ToolRegistry()
        if tools:
            for t in tools:
                self.registry.register(t)
        self.client = None if USE_MOCK_LLM else OpenAI()
    
    def _system_prompt(self):
        return f"""You are a helpful assistant with tools.

{self.registry.format_for_prompt()}

Format:
Thought: [reasoning]
Action: [tool name]
Action Input: [input]

Or when done:
Thought: [reasoning]
Final Answer: [answer]"""
    
    def _parse(self, text):
        result = {'thought': None, 'action': None, 'action_input': None, 'final_answer': None}
        m = re.search(r'Thought:\s*(.+?)(?=\n(?:Action|Final Answer)|$)', text, re.DOTALL)
        if m: result['thought'] = m.group(1).strip()
        m = re.search(r'Final Answer:\s*(.+)', text, re.DOTALL)
        if m: result['final_answer'] = m.group(1).strip(); return result
        m = re.search(r'Action:\s*(.+?)(?=\n|$)', text)
        if m: result['action'] = m.group(1).strip()
        m = re.search(r'Action Input:\s*(.+?)(?=\n(?:Thought|Observation)|$)', text, re.DOTALL)
        if m: result['action_input'] = m.group(1).strip()
        return result
    
    def _exec_tool(self, action, action_input):
        tool = self.registry.get(action)
        if not tool: return f"Error: Tool '{action}' not found."
        props = list(tool.parameters.get("properties", {}).keys())
        try:
            return tool.execute(**{props[0]: action_input}) if props else tool.execute()
        except Exception as e:
            return f"Error: {e}"
    
    def _call_llm(self, messages):
        if USE_MOCK_LLM:
            return self._mock(messages)
        r = self.client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0, max_tokens=1024)
        return r.choices[0].message.content
    
    def _mock(self, messages):
        obs_count = sum(1 for m in messages if "Observation:" in m.get("content", ""))
        user_query = next((m["content"] for m in messages if m["role"] == "user"), "")
        if obs_count == 0:
            if "search" in user_query.lower() or "find" in user_query.lower() or "research" in user_query.lower():
                topic = user_query.split("about")[-1].strip() if "about" in user_query else user_query[:50]
                return f"Thought: I need to search for information.\nAction: Search\nAction Input: {topic}"
            elif "calculate" in user_query.lower() or "compute" in user_query.lower():
                return "Thought: I need to calculate this.\nAction: Calculator\nAction Input: 42"
            else:
                return f"Thought: Let me search for relevant information.\nAction: Search\nAction Input: {user_query[:60]}"
        else:
            return f"Thought: I have the information I need.\nFinal Answer: Based on my research, here is a summary of the findings from the search results above. The key points are covered in the observations."
    
    def run(self, query: str) -> str:
        msgs = [{"role": "system", "content": self._system_prompt()}, {"role": "user", "content": query}]
        for step in range(self.max_steps):
            resp = self._call_llm(msgs)
            parsed = self._parse(resp)
            if self.verbose:
                print(f"  [Step {step+1}] Thought: {(parsed['thought'] or 'N/A')[:80]}")
            if parsed['final_answer']:
                return parsed['final_answer']
            if parsed['action'] and parsed['action_input'] is not None:
                obs = self._exec_tool(parsed['action'], parsed['action_input'])
                if self.verbose:
                    print(f"           Action: {parsed['action']}({parsed['action_input'][:40]}) -> {obs[:60]}")
                msgs.append({"role": "assistant", "content": resp})
                msgs.append({"role": "user", "content": f"Observation: {obs}"})
            else:
                msgs.append({"role": "assistant", "content": resp})
                msgs.append({"role": "user", "content": "Please use the Thought/Action/Action Input format or give a Final Answer."})
        return "Max steps reached."

# Quick test
test_agent = SimpleReActAgent(tools=[CalculatorTool(), SearchTool(), PythonREPLTool()], verbose=True)
print(test_agent.run("Search for information about RLHF"))

---
## 3. Planning: Why Agents Need It

### When ReAct isn't enough

The ReAct loop works for simple tasks: search → calculate → answer. But consider:

> "Write a research summary comparing RLHF and DPO, including their mathematical foundations, practical trade-offs, and recent developments."

This requires:
1. Searching for RLHF papers and key concepts
2. Searching for DPO papers and key concepts
3. Understanding the math behind both
4. Identifying practical trade-offs
5. Finding recent developments
6. Synthesizing everything into a coherent summary

A reactive agent will get lost — it has no big-picture plan, just reacts to each observation. Planning solves this by **decomposing the goal into ordered subtasks** before execution.

### Planning = Turning a vague goal into concrete steps

```
Goal: "Compare RLHF and DPO"
  ↓ (task decomposition)
Plan:
  1. Search for RLHF overview and key papers
  2. Search for DPO overview and key papers
  3. Extract mathematical formulations of both
  4. Identify practical trade-offs (cost, quality, complexity)
  5. Search for recent developments and comparisons
  6. Synthesize into structured summary
```

---
## 4. Task Decomposition

In [ ]:
@dataclass
class Subtask:
    """A single subtask in a plan."""
    id: int
    description: str
    dependencies: List[int] = field(default_factory=list)  # IDs of subtasks this depends on
    expected_output: str = ""
    status: str = "pending"  # pending, running, completed, failed, skipped
    result: Optional[str] = None
    is_optional: bool = False


class TaskDecomposer:
    """Decomposes complex tasks into ordered subtasks using an LLM.
    
    The decomposer prompts the LLM to break a complex goal into:
    - Ordered subtasks with clear descriptions
    - Dependencies (which subtasks must complete first)
    - Expected outputs (what each subtask should produce)
    """
    
    def __init__(self, model: str = "gpt-4o-mini"):
        self.model = model
        self.client = None if USE_MOCK_LLM else OpenAI()
    
    def decompose(self, task: str) -> List[Subtask]:
        """Decompose a complex task into subtasks."""
        if USE_MOCK_LLM:
            return self._mock_decompose(task)
        
        prompt = f"""Decompose this task into 4-7 ordered subtasks.

Task: {task}

For each subtask provide:
- id: sequential number starting from 1
- description: what to do (be specific and actionable)
- dependencies: list of subtask IDs that must complete first (empty list if none)
- expected_output: what this subtask should produce
- is_optional: true if this subtask can be skipped without breaking the plan

Respond in JSON format:
[{{"id": 1, "description": "...", "dependencies": [], "expected_output": "...", "is_optional": false}}, ...]"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"},
        )
        
        data = json.loads(response.choices[0].message.content)
        # Handle both formats: list directly or wrapped in an object
        subtask_list = data if isinstance(data, list) else data.get("subtasks", data.get("tasks", []))
        
        return [
            Subtask(
                id=s["id"],
                description=s["description"],
                dependencies=s.get("dependencies", []),
                expected_output=s.get("expected_output", ""),
                is_optional=s.get("is_optional", False),
            )
            for s in subtask_list
        ]
    
    def _mock_decompose(self, task: str) -> List[Subtask]:
        """Mock decomposition for demonstration."""
        task_lower = task.lower()
        
        if "rlhf" in task_lower or "dpo" in task_lower or "research summary" in task_lower:
            return [
                Subtask(id=1, description="Search for RLHF overview, key papers, and core methodology",
                        dependencies=[], expected_output="Summary of RLHF: papers, process, key insights"),
                Subtask(id=2, description="Search for DPO overview, key papers, and how it simplifies RLHF",
                        dependencies=[], expected_output="Summary of DPO: papers, process, key insights"),
                Subtask(id=3, description="Extract and compare the mathematical formulations of RLHF (PPO-based) and DPO (classification-based)",
                        dependencies=[1, 2], expected_output="Mathematical comparison of both approaches"),
                Subtask(id=4, description="Identify practical trade-offs: training cost, data requirements, implementation complexity, alignment quality",
                        dependencies=[1, 2], expected_output="Trade-off analysis table"),
                Subtask(id=5, description="Search for recent developments and variants (KTO, IPO, ORPO, etc.)",
                        dependencies=[], expected_output="Recent developments summary", is_optional=True),
                Subtask(id=6, description="Synthesize all findings into a structured research summary with introduction, comparison, and conclusions",
                        dependencies=[1, 2, 3, 4], expected_output="Complete research summary document"),
            ]
        else:
            return [
                Subtask(id=1, description=f"Research the main topic: {task[:80]}",
                        dependencies=[], expected_output="Initial research findings"),
                Subtask(id=2, description="Gather supporting details and evidence",
                        dependencies=[1], expected_output="Supporting evidence"),
                Subtask(id=3, description="Analyze and organize findings",
                        dependencies=[1, 2], expected_output="Organized analysis"),
                Subtask(id=4, description="Produce final output",
                        dependencies=[3], expected_output="Final result"),
            ]


# Demo
decomposer = TaskDecomposer()
subtasks = decomposer.decompose(
    "Write a research summary comparing RLHF and DPO, including their "
    "mathematical foundations, practical trade-offs, and recent developments."
)

print("TASK DECOMPOSITION")
print("=" * 60)
for st in subtasks:
    opt = " [OPTIONAL]" if st.is_optional else ""
    deps = f" (depends on: {st.dependencies})" if st.dependencies else ""
    print(f"\n  {st.id}. {st.description}{opt}{deps}")
    print(f"     Expected: {st.expected_output}")

In [ ]:
class PlanExecutor:
    """Executes a plan (list of subtasks) using a ReAct agent.
    
    Features:
    - Respects dependency ordering
    - Handles failures: re-plan, skip optional subtasks
    - Passes context from completed subtasks to subsequent ones
    """
    
    def __init__(self, agent: SimpleReActAgent, verbose: bool = True):
        self.agent = agent
        self.verbose = verbose
    
    def _can_execute(self, subtask: Subtask, completed: set) -> bool:
        """Check if all dependencies are satisfied."""
        return all(dep_id in completed for dep_id in subtask.dependencies)
    
    def _build_context(self, subtask: Subtask, all_subtasks: List[Subtask]) -> str:
        """Build context from completed subtasks for the current subtask."""
        context_parts = []
        for dep_id in subtask.dependencies:
            dep = next((s for s in all_subtasks if s.id == dep_id), None)
            if dep and dep.result:
                context_parts.append(
                    f"[Result from subtask {dep.id} ({dep.description[:50]})]:\n{dep.result[:500]}"
                )
        return "\n\n".join(context_parts)
    
    def execute(self, subtasks: List[Subtask], overall_goal: str = "") -> Dict[str, Any]:
        """Execute all subtasks in dependency order."""
        completed = set()
        failed = set()
        results = {}
        
        if self.verbose:
            print(f"\n{'='*60}")
            print(f"EXECUTING PLAN ({len(subtasks)} subtasks)")
            print(f"Goal: {overall_goal}")
            print(f"{'='*60}")
        
        # Simple topological execution (assumes subtasks are already ordered)
        max_iterations = len(subtasks) * 2  # Safety bound
        iteration = 0
        
        while len(completed) + len(failed) < len(subtasks) and iteration < max_iterations:
            iteration += 1
            progress_made = False
            
            for subtask in subtasks:
                if subtask.id in completed or subtask.id in failed:
                    continue
                
                if not self._can_execute(subtask, completed):
                    # Check if any dependency failed
                    if any(dep_id in failed for dep_id in subtask.dependencies):
                        if subtask.is_optional:
                            subtask.status = "skipped"
                            failed.add(subtask.id)
                            if self.verbose:
                                print(f"\n  [{subtask.id}] SKIPPED (optional, dependency failed): {subtask.description[:60]}")
                        else:
                            subtask.status = "failed"
                            failed.add(subtask.id)
                            if self.verbose:
                                print(f"\n  [{subtask.id}] FAILED (dependency failed): {subtask.description[:60]}")
                        progress_made = True
                    continue
                
                # Execute the subtask
                subtask.status = "running"
                if self.verbose:
                    print(f"\n  [{subtask.id}] RUNNING: {subtask.description}")
                
                # Build the query with context from dependencies
                context = self._build_context(subtask, subtasks)
                query = subtask.description
                if context:
                    query = f"{subtask.description}\n\nContext from previous steps:\n{context}"
                
                try:
                    result = self.agent.run(query)
                    subtask.result = result
                    subtask.status = "completed"
                    completed.add(subtask.id)
                    results[subtask.id] = result
                    if self.verbose:
                        print(f"  [{subtask.id}] COMPLETED: {result[:100]}...")
                    progress_made = True
                    
                except Exception as e:
                    if subtask.is_optional:
                        subtask.status = "skipped"
                        if self.verbose:
                            print(f"  [{subtask.id}] SKIPPED (optional, error): {e}")
                    else:
                        subtask.status = "failed"
                        subtask.result = f"Error: {e}"
                        if self.verbose:
                            print(f"  [{subtask.id}] FAILED: {e}")
                    failed.add(subtask.id)
                    progress_made = True
            
            if not progress_made:
                if self.verbose:
                    print("\n  No progress — possible circular dependencies. Aborting.")
                break
        
        return {
            "completed": len(completed),
            "failed": len(failed),
            "total": len(subtasks),
            "results": results,
            "subtasks": subtasks,
        }


print("PlanExecutor defined.")

In [ ]:
# Execute the plan we decomposed above
executor = PlanExecutor(
    agent=SimpleReActAgent(
        tools=[CalculatorTool(), SearchTool(), PythonREPLTool()],
        verbose=True,
    ),
    verbose=True,
)

plan_result = executor.execute(
    subtasks=subtasks,
    overall_goal="Compare RLHF and DPO"
)

print(f"\n{'='*60}")
print(f"PLAN EXECUTION SUMMARY")
print(f"Completed: {plan_result['completed']}/{plan_result['total']}")
print(f"Failed: {plan_result['failed']}/{plan_result['total']}")

---
## 5. Tree of Thoughts

**Chain-of-Thought (CoT)**: one reasoning path, straight line, no backtracking.  
**Tree of Thoughts (ToT)**: multiple reasoning paths, evaluate each, select the best, backtrack if needed.

```
CoT:    A → B → C → D  (linear, hope for the best)

ToT:         A
           / | \
          B1  B2  B3      (generate multiple candidates)
         /|   |    |\
        C1 C2 C3  C4 C5   (evaluate, prune, expand best)
           ↑
       (best path)
```

Key insight from Yao et al. (2023): LLMs can serve as both the **generator** (propose thoughts) and the **evaluator** (rate how promising each path is).

When to use ToT:
- Problems with multiple valid approaches (creative tasks, puzzle solving)
- When the first approach might fail and you need to backtrack
- When you can evaluate intermediate progress

When NOT to use:
- Simple, well-defined tasks (overkill)
- When evaluation is impossible or unreliable
- When cost matters (ToT uses many more LLM calls)

In [ ]:
@dataclass
class ThoughtNode:
    """A node in the tree of thoughts."""
    id: str
    thought: str
    score: float = 0.0
    parent_id: Optional[str] = None
    children: List[str] = field(default_factory=list)
    depth: int = 0


class TreeOfThoughts:
    """Tree of Thoughts: explore multiple reasoning paths with evaluation and backtracking.
    
    Architecture:
    1. generate_thoughts(): propose k possible next steps from current state
    2. evaluate_thoughts(): score each candidate (0-1)
    3. search(): BFS or DFS through the thought tree
    
    The LLM serves as both generator and evaluator.
    """
    
    def __init__(self, model: str = "gpt-4o-mini", max_depth: int = 4, branching_factor: int = 3):
        self.model = model
        self.max_depth = max_depth
        self.branching_factor = branching_factor  # k in the paper
        self.client = None if USE_MOCK_LLM else OpenAI()
        self.nodes: Dict[str, ThoughtNode] = {}
        self._node_counter = 0
    
    def _new_id(self) -> str:
        self._node_counter += 1
        return f"node_{self._node_counter}"
    
    def _get_path(self, node_id: str) -> List[str]:
        """Get the reasoning path from root to this node."""
        path = []
        current = node_id
        while current is not None:
            path.append(self.nodes[current].thought)
            current = self.nodes[current].parent_id
        return list(reversed(path))
    
    def generate_thoughts(self, problem: str, state: List[str], k: int = 3) -> List[str]:
        """Generate k possible next thoughts given the current reasoning state."""
        if USE_MOCK_LLM:
            return self._mock_generate(problem, state, k)
        
        state_text = "\n".join(f"Step {i+1}: {s}" for i, s in enumerate(state))
        prompt = f"""Problem: {problem}

Current reasoning path:
{state_text if state else '(empty - this is the first step)'}

Generate exactly {k} different possible next steps. Each should be a distinct approach or continuation.
Format: Return a JSON list of {k} strings, each being one possible next thought.
Example: ["First approach...", "Alternative approach...", "Third possibility..."]"""
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,  # Higher temp for diversity
            response_format={"type": "json_object"},
        )
        
        data = json.loads(response.choices[0].message.content)
        thoughts = data if isinstance(data, list) else data.get("thoughts", data.get("steps", []))
        return thoughts[:k]
    
    def evaluate_thoughts(self, problem: str, state: List[str], thoughts: List[str]) -> List[float]:
        """Evaluate each thought: how promising is this path? Returns scores 0-1."""
        if USE_MOCK_LLM:
            return self._mock_evaluate(thoughts)
        
        state_text = "\n".join(f"Step {i+1}: {s}" for i, s in enumerate(state))
        
        scores = []
        for thought in thoughts:
            prompt = f"""Problem: {problem}

Reasoning so far:
{state_text if state else '(first step)'}

Proposed next step: {thought}

Rate this step on a scale of 0.0 to 1.0:
- 1.0 = excellent, clearly moves toward the solution
- 0.5 = okay, might work but uncertain
- 0.0 = bad, clearly wrong or unhelpful

Respond with just a number."""
            
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=10,
            )
            
            try:
                score = float(re.search(r'[0-9.]+', response.choices[0].message.content).group())
                scores.append(min(1.0, max(0.0, score)))
            except:
                scores.append(0.5)  # Default if parsing fails
        
        return scores
    
    def _mock_generate(self, problem: str, state: List[str], k: int) -> List[str]:
        """Mock thought generation."""
        depth = len(state)
        if depth == 0:
            return [
                "Start by identifying the key components and constraints of the problem.",
                "Begin with a simple example to build intuition before tackling the full problem.",
                "Break the problem into smaller sub-problems that can be solved independently.",
            ][:k]
        elif depth == 1:
            return [
                "Apply the identified approach systematically to each component.",
                "Look for patterns or symmetries that can simplify the solution.",
                "Consider edge cases that might break the current approach.",
            ][:k]
        else:
            return [
                "Verify the solution by checking against known constraints.",
                "Optimize the solution for efficiency if possible.",
                "Synthesize the findings into a final answer.",
            ][:k]
    
    def _mock_evaluate(self, thoughts: List[str]) -> List[float]:
        """Mock thought evaluation."""
        scores = []
        for i, t in enumerate(thoughts):
            # Slightly prefer the first thought (deterministic for demo)
            base = 0.8 - 0.1 * i
            if "verify" in t.lower() or "break" in t.lower() or "identify" in t.lower():
                base += 0.1
            scores.append(min(1.0, max(0.0, base)))
        return scores
    
    def search(self, problem: str, strategy: str = "bfs") -> Dict[str, Any]:
        """Search through the tree of thoughts.
        
        Args:
            problem: The problem to solve
            strategy: 'bfs' (breadth-first) or 'dfs' (depth-first)
        
        Returns:
            dict with best_path, all_nodes, and search stats
        """
        self.nodes = {}
        self._node_counter = 0
        
        # Create root node
        root_id = self._new_id()
        self.nodes[root_id] = ThoughtNode(id=root_id, thought=problem, score=1.0, depth=0)
        
        if strategy == "bfs":
            return self._bfs(problem, root_id)
        else:
            return self._dfs(problem, root_id)
    
    def _bfs(self, problem: str, root_id: str) -> Dict[str, Any]:
        """Breadth-first search through thought tree."""
        frontier = [root_id]
        best_leaf = root_id
        # NOTE: best_score semantics differ by strategy -- BFS tracks the best
        # SINGLE-NODE score (0-1), while DFS tracks a CUMULATIVE path score.
        best_score = 0.0
        stats = {"nodes_explored": 0, "nodes_pruned": 0}
        
        for depth in range(self.max_depth):
            if not frontier:
                break
            
            next_frontier = []
            
            for node_id in frontier:
                state = self._get_path(node_id)
                stats["nodes_explored"] += 1
                
                # Generate candidate thoughts
                thoughts = self.generate_thoughts(problem, state, self.branching_factor)
                
                # Evaluate them
                scores = self.evaluate_thoughts(problem, state, thoughts)
                
                # Create child nodes
                for thought, score in zip(thoughts, scores):
                    child_id = self._new_id()
                    child = ThoughtNode(
                        id=child_id,
                        thought=thought,
                        score=score,
                        parent_id=node_id,
                        depth=depth + 1,
                    )
                    self.nodes[child_id] = child
                    self.nodes[node_id].children.append(child_id)
                    
                    # Prune low-scoring paths
                    if score >= 0.5:
                        next_frontier.append(child_id)
                        if score > best_score:
                            best_score = score
                            best_leaf = child_id
                    else:
                        stats["nodes_pruned"] += 1
            
            # Keep only top candidates for next level
            next_frontier.sort(key=lambda nid: self.nodes[nid].score, reverse=True)
            frontier = next_frontier[:self.branching_factor]  # Beam width
        
        best_path = self._get_path(best_leaf)
        stats["total_nodes"] = len(self.nodes)
        
        return {
            "best_path": best_path,
            "best_score": best_score,
            "stats": stats,
        }
    
    def _dfs(self, problem: str, root_id: str) -> Dict[str, Any]:
        """Depth-first search through thought tree."""
        best_path = []
        # NOTE: here best_score is a CUMULATIVE path score (sum of node scores along
        # the path), unlike BFS's best single-node score -- the two are not comparable.
        best_score = 0.0
        stats = {"nodes_explored": 0, "nodes_pruned": 0, "backtrack_count": 0}
        
        def _dfs_recurse(node_id: str, current_score: float):
            nonlocal best_path, best_score
            
            node = self.nodes[node_id]
            stats["nodes_explored"] += 1
            
            if node.depth >= self.max_depth:
                path = self._get_path(node_id)
                if current_score > best_score:
                    best_score = current_score
                    best_path = path
                return
            
            state = self._get_path(node_id)
            thoughts = self.generate_thoughts(problem, state, self.branching_factor)
            scores = self.evaluate_thoughts(problem, state, thoughts)
            
            # Sort by score (explore best first in DFS)
            candidates = sorted(zip(thoughts, scores), key=lambda x: x[1], reverse=True)
            
            for thought, score in candidates:
                if score < 0.3:  # Pruning threshold
                    stats["nodes_pruned"] += 1
                    continue
                
                child_id = self._new_id()
                child = ThoughtNode(
                    id=child_id, thought=thought, score=score,
                    parent_id=node_id, depth=node.depth + 1,
                )
                self.nodes[child_id] = child
                node.children.append(child_id)
                
                _dfs_recurse(child_id, current_score + score)
                stats["backtrack_count"] += 1  # Each return is a backtrack
        
        _dfs_recurse(root_id, 0.0)
        stats["total_nodes"] = len(self.nodes)
        
        return {
            "best_path": best_path,
            "best_score": best_score,
            "stats": stats,
        }


print("TreeOfThoughts defined.")

In [ ]:
# Demo: Solve a problem using Tree of Thoughts
tot = TreeOfThoughts(max_depth=3, branching_factor=3)

problem = (
    "Design a system to detect and mitigate hallucinations in LLM outputs "
    "in real-time for a production chatbot serving 1M users."
)

# BFS search
print("=" * 60)
print("TREE OF THOUGHTS — BFS")
print("=" * 60)
bfs_result = tot.search(problem, strategy="bfs")

print(f"\nBest path (score: {bfs_result['best_score']:.2f}):")
for i, step in enumerate(bfs_result['best_path']):
    prefix = "Problem" if i == 0 else f"Step {i}"
    print(f"  {prefix}: {step}")
print(f"\nStats: {bfs_result['stats']}")

# DFS search
print(f"\n{'='*60}")
print("TREE OF THOUGHTS — DFS")
print("=" * 60)
tot2 = TreeOfThoughts(max_depth=3, branching_factor=3)
dfs_result = tot2.search(problem, strategy="dfs")

print(f"\nBest path (score: {dfs_result['best_score']:.2f}):")
for i, step in enumerate(dfs_result['best_path']):
    prefix = "Problem" if i == 0 else f"Step {i}"
    print(f"  {prefix}: {step}")
print(f"\nStats: {dfs_result['stats']}")

print(f"\n{'='*60}")
print("COMPARISON")
print(f"BFS: {bfs_result['stats']['nodes_explored']} explored, {bfs_result['stats']['nodes_pruned']} pruned")
print(f"DFS: {dfs_result['stats']['nodes_explored']} explored, {dfs_result['stats']['nodes_pruned']} pruned")

---
## 6. Memory Systems

Agents need memory to be effective across multi-step tasks and sessions.

### Three Types of Agent Memory

| Type | What it stores | Implementation | Analogy |
|------|---------------|----------------|----------|
| **Short-term** | Current conversation context | The LLM's context window | Working memory |
| **Long-term** | Facts, knowledge, documents | Vector database (FAISS, Pinecone) | Reference library |
| **Episodic** | Past task attempts + outcomes | Structured store with similarity search | Personal experience |

### Why This Matters

- **Short-term memory is limited**: Context windows are 128K+ tokens now, but agents can easily fill them with observations. You need to manage this.
- **Long-term memory enables recall**: Agent can access information from past interactions or a knowledge base.
- **Episodic memory enables learning**: Agent remembers what worked and what didn't. "I tried approach X for a similar problem and it failed because Y."

**Insider Tip:** Vector databases for agent memory are overhyped in interviews. What matters more is: (1) knowing WHEN to retrieve vs compute, (2) managing context window budget, (3) summarization strategies for long histories. Show awareness of the practical trade-offs.

---
## 7. Long-Term Memory with Vector Store

In [ ]:
class VectorMemory:
    """Long-term memory using FAISS vector store + sentence-transformers.
    
    This provides semantic search over stored information — the agent
    can retrieve relevant past observations by meaning, not just keywords.
    
    Components:
    - Embedding model: converts text to vectors
    - FAISS index: efficient similarity search
    - Metadata store: text + metadata alongside vectors
    """
    
    def __init__(self, embedding_model: str = "all-MiniLM-L6-v2", use_mock: bool = False):
        self.use_mock = use_mock
        self.texts: List[str] = []
        self.metadata: List[Dict[str, Any]] = []
        self.timestamps: List[float] = []
        
        if use_mock:
            # Mock mode: use random vectors for demonstration
            self.dimension = 384
            self.embedder = None
            print("VectorMemory initialized in mock mode (random vectors).")
        else:
            try:
                from sentence_transformers import SentenceTransformer
                self.embedder = SentenceTransformer(embedding_model)
                self.dimension = self.embedder.get_sentence_embedding_dimension()
                print(f"VectorMemory initialized with {embedding_model} (dim={self.dimension}).")
            except Exception as e:
                print(f"Could not load embedding model: {e}")
                print("Falling back to mock mode.")
                self.use_mock = True
                self.dimension = 384
                self.embedder = None
        
        import faiss
        self.index = faiss.IndexFlatIP(self.dimension)  # Inner product (cosine sim with normalized vectors)
    
    def _embed(self, texts: List[str]) -> np.ndarray:
        """Embed texts into vectors."""
        if self.use_mock:
            # Generate deterministic pseudo-random vectors based on text hash
            vectors = []
            for text in texts:
                seed = int(hashlib.md5(text.encode()).hexdigest()[:8], 16)
                rng = np.random.RandomState(seed)
                vec = rng.randn(self.dimension).astype(np.float32)
                vec = vec / np.linalg.norm(vec)  # Normalize
                vectors.append(vec)
            return np.array(vectors)
        else:
            embeddings = self.embedder.encode(texts, normalize_embeddings=True)
            return embeddings.astype(np.float32)
    
    def store(self, text: str, metadata: Optional[Dict[str, Any]] = None) -> int:
        """Store a text with optional metadata. Returns the index."""
        embedding = self._embed([text])
        self.index.add(embedding)
        self.texts.append(text)
        self.metadata.append(metadata or {})
        self.timestamps.append(time.time())
        return len(self.texts) - 1
    
    def store_batch(self, texts: List[str], metadatas: Optional[List[Dict]] = None) -> List[int]:
        """Store multiple texts at once (more efficient)."""
        embeddings = self._embed(texts)
        start_idx = len(self.texts)
        self.index.add(embeddings)
        self.texts.extend(texts)
        self.metadata.extend(metadatas or [{} for _ in texts])
        self.timestamps.extend([time.time()] * len(texts))
        return list(range(start_idx, start_idx + len(texts)))
    
    def retrieve(self, query: str, k: int = 5) -> List[Dict[str, Any]]:
        """Retrieve k most similar stored items."""
        if self.index.ntotal == 0:
            return []
        
        query_vec = self._embed([query])
        k = min(k, self.index.ntotal)
        scores, indices = self.index.search(query_vec, k)
        
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:  # FAISS returns -1 for invalid results
                continue
            results.append({
                "text": self.texts[idx],
                "score": float(score),
                "metadata": self.metadata[idx],
                "index": int(idx),
            })
        
        return results
    
    def forget(self, max_age_seconds: Optional[float] = None, score_threshold: Optional[float] = None) -> int:
        """Remove old or irrelevant items.
        
        Note: FAISS doesn't support deletion easily. We rebuild the index.
        In production, use a vector DB that supports deletion (Pinecone, Weaviate, etc.)
        """
        now = time.time()
        keep_indices = []
        
        for i in range(len(self.texts)):
            if max_age_seconds and (now - self.timestamps[i]) > max_age_seconds:
                continue  # Too old, forget it
            keep_indices.append(i)
        
        removed = len(self.texts) - len(keep_indices)
        
        if removed > 0:
            # Rebuild with kept items
            kept_texts = [self.texts[i] for i in keep_indices]
            kept_metadata = [self.metadata[i] for i in keep_indices]
            kept_timestamps = [self.timestamps[i] for i in keep_indices]
            
            self.texts = kept_texts
            self.metadata = kept_metadata
            self.timestamps = kept_timestamps
            
            import faiss
            self.index = faiss.IndexFlatIP(self.dimension)
            if self.texts:
                embeddings = self._embed(self.texts)
                self.index.add(embeddings)
        
        return removed
    
    def __len__(self):
        return len(self.texts)


print("VectorMemory defined.")

In [ ]:
# Demo: Build and query a vector memory
memory = VectorMemory(use_mock=True)  # Set to False to use real embeddings

# Store some agent observations
observations = [
    ("RLHF uses a three-step process: SFT, reward model training, and PPO optimization.",
     {"source": "search", "task": "RLHF research", "step": 1}),
    ("DPO simplifies RLHF by using a classification loss directly on preference pairs.",
     {"source": "search", "task": "DPO research", "step": 2}),
    ("The main trade-off: RLHF is more flexible but harder to train; DPO is simpler but less expressive.",
     {"source": "analysis", "task": "comparison", "step": 3}),
    ("Flash Attention reduces memory usage from O(N^2) to O(N) using tiling.",
     {"source": "search", "task": "efficiency research", "step": 1}),
    ("The transformer architecture processes sequences in parallel using self-attention.",
     {"source": "search", "task": "architecture research", "step": 1}),
    ("Multi-agent debate improves accuracy by having agents argue and reach consensus.",
     {"source": "search", "task": "multi-agent research", "step": 1}),
    ("KTO (Kahneman-Tversky Optimization) uses binary feedback instead of pairwise preferences.",
     {"source": "search", "task": "alignment variants", "step": 1}),
]

for text, meta in observations:
    memory.store(text, meta)

print(f"Stored {len(memory)} observations in memory.\n")

# Query the memory
queries = [
    "How does preference learning work?",
    "What are efficient attention mechanisms?",
    "How do multiple agents collaborate?",
]

for query in queries:
    print(f"Query: '{query}'")
    results = memory.retrieve(query, k=3)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['text'][:80]}...")
    print()

---
## 8. Episodic Memory

In [ ]:
@dataclass
class Episode:
    """A record of a past task attempt."""
    task: str
    plan: List[str]  # The subtasks attempted
    outcome: str  # "success" or "failure"
    result: Optional[str] = None
    lessons_learned: List[str] = field(default_factory=list)
    duration_ms: float = 0.0
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class EpisodicMemory:
    """Stores past task attempts and their outcomes.
    
    Before planning a new task, the agent can retrieve similar past tasks
    and learn from their outcomes:
    - "I tried approach X for a similar problem and it failed because Y"
    - "This type of task works best with approach Z"
    
    Uses the same VectorMemory for semantic search over task descriptions.
    """
    
    def __init__(self, use_mock: bool = True):
        self.episodes: List[Episode] = []
        self.vector_store = VectorMemory(use_mock=use_mock)
    
    def record_episode(
        self,
        task: str,
        plan: List[str],
        outcome: str,
        result: Optional[str] = None,
        lessons_learned: Optional[List[str]] = None,
        duration_ms: float = 0.0,
    ) -> int:
        """Record a completed task attempt."""
        episode = Episode(
            task=task,
            plan=plan,
            outcome=outcome,
            result=result,
            lessons_learned=lessons_learned or [],
            duration_ms=duration_ms,
        )
        idx = len(self.episodes)
        self.episodes.append(episode)
        
        # Store in vector memory for semantic retrieval
        summary = (
            f"Task: {task}\n"
            f"Outcome: {outcome}\n"
            f"Lessons: {'; '.join(lessons_learned or [])}"
        )
        self.vector_store.store(summary, {"episode_idx": idx, "outcome": outcome})
        
        return idx
    
    def recall_similar(self, task: str, k: int = 3) -> List[Dict[str, Any]]:
        """Find similar past tasks and their outcomes."""
        results = self.vector_store.retrieve(task, k=k)
        
        recalled = []
        for r in results:
            ep_idx = r["metadata"].get("episode_idx")
            if ep_idx is not None and ep_idx < len(self.episodes):
                ep = self.episodes[ep_idx]
                recalled.append({
                    "task": ep.task,
                    "outcome": ep.outcome,
                    "plan": ep.plan,
                    "lessons": ep.lessons_learned,
                    "similarity": r["score"],
                })
        
        return recalled
    
    def get_lessons_for_task(self, task: str, k: int = 3) -> str:
        """Get a formatted string of lessons from similar past tasks."""
        similar = self.recall_similar(task, k)
        
        if not similar:
            return "No similar past tasks found."
        
        parts = ["Lessons from similar past tasks:"]
        for s in similar:
            parts.append(f"\n  Task: {s['task'][:60]}")
            parts.append(f"  Outcome: {s['outcome']}")
            if s['lessons']:
                for lesson in s['lessons']:
                    parts.append(f"    - {lesson}")
        
        return "\n".join(parts)


print("EpisodicMemory defined.")

In [ ]:
# Demo: Episodic memory in action
episodic = EpisodicMemory(use_mock=True)

# Record some past task attempts
episodic.record_episode(
    task="Compare RLHF and DPO for language model alignment",
    plan=["Search RLHF papers", "Search DPO papers", "Compare mathematically", "Write summary"],
    outcome="success",
    result="Comprehensive comparison showing DPO is simpler but RLHF is more flexible.",
    lessons_learned=[
        "Start with the mathematical formulation — it makes the comparison clearer",
        "Include practical considerations (cost, data) not just theory",
        "The search tool returns good results for specific paper names",
    ],
)

episodic.record_episode(
    task="Summarize recent advances in efficient attention mechanisms",
    plan=["Search for Flash Attention", "Search for linear attention", "Compare approaches"],
    outcome="partial",
    lessons_learned=[
        "Search results were too broad — use specific paper names",
        "Need to include complexity analysis (O notation) for fair comparison",
        "Missing recent papers (2024) — the search tool's knowledge cutoff limits this",
    ],
)

episodic.record_episode(
    task="Build a multi-agent debate system",
    plan=["Design agent roles", "Implement debate protocol", "Run debate", "Evaluate consensus"],
    outcome="failure",
    lessons_learned=[
        "Two agents converged too quickly — need diversity pressure",
        "Turn limit was too low (3 turns) — increase to 5-7",
        "Agents need different system prompts to maintain distinct perspectives",
    ],
)

print(f"Recorded {len(episodic.episodes)} episodes.\n")

# Now query: what should I know before doing a similar task?
new_task = "Compare PPO and DPO for preference optimization"
print(f"New task: {new_task}")
print(episodic.get_lessons_for_task(new_task))

---
## 9. Working Memory Management

In [ ]:
class WorkingMemoryManager:
    """Manages the agent's working memory (context window).
    
    Problem: Each agent step adds tokens. After 5-10 steps, the context
    window fills up and the agent loses track of earlier information.
    
    Solution: Summarize old observations, keep recent ones in full,
    and maintain a priority queue of important information.
    
    Token estimation: ~4 characters per token (rough GPT estimate)
    """
    
    def __init__(self, max_tokens: int = 8000, summary_threshold: float = 0.7):
        self.max_tokens = max_tokens
        self.summary_threshold = summary_threshold  # Summarize when usage exceeds this fraction
        self.messages: List[Dict[str, str]] = []
        self.system_prompt: str = ""
        self.summaries: List[str] = []  # Running summaries of old context
    
    def _estimate_tokens(self, text: str) -> int:
        """Rough token count estimation (~4 chars per token)."""
        return len(text) // 4
    
    def _total_tokens(self) -> int:
        """Estimate total tokens in current messages."""
        total = self._estimate_tokens(self.system_prompt)
        for msg in self.messages:
            total += self._estimate_tokens(msg["content"])
        return total
    
    def set_system_prompt(self, prompt: str):
        """Set the system prompt (always kept)."""
        self.system_prompt = prompt
    
    def add_message(self, role: str, content: str):
        """Add a message and check if summarization is needed."""
        self.messages.append({"role": role, "content": content})
        
        # Check if we need to summarize
        usage = self._total_tokens() / self.max_tokens
        if usage > self.summary_threshold:
            self._summarize_old_messages()
    
    def _summarize_old_messages(self):
        """Summarize the oldest messages to free up context space.
        
        Strategy:
        - Keep the first user message (the original query)
        - Keep the last 4 messages (recent context)
        - Summarize everything in between
        """
        if len(self.messages) <= 5:
            return  # Not enough messages to summarize
        
        # Split: first message + middle (to summarize) + recent
        first_msg = self.messages[0]  # Original query
        middle = self.messages[1:-4]  # Old messages to summarize
        recent = self.messages[-4:]   # Keep recent in full
        
        if not middle:
            return
        
        # Create summary of middle messages
        summary_parts = []
        for msg in middle:
            role = msg["role"]
            content = msg["content"]
            if content.startswith("[Summary of earlier steps]"):
                # Carry prior summaries forward -- otherwise they match no branch below and are silently dropped
                summary_parts.append(content.split("\n", 1)[1] if "\n" in content else content)
            elif "Observation:" in content:
                # Extract key observation
                obs = content.replace("Observation:", "").strip()
                summary_parts.append(f"- Observed: {obs[:150]}")
            elif role == "assistant":
                # Extract thought and action
                thought_match = re.search(r'Thought:\s*(.+?)(?=\n|$)', content)
                action_match = re.search(r'Action:\s*(.+?)(?=\n|$)', content)
                if thought_match:
                    summary_parts.append(f"- Thought: {thought_match.group(1).strip()[:100]}")
                if action_match:
                    summary_parts.append(f"- Action: {action_match.group(1).strip()}")
        
        summary = "[Summary of earlier steps]\n" + "\n".join(summary_parts)
        self.summaries.append(summary)
        
        # Reconstruct messages
        self.messages = [
            first_msg,
            {"role": "user", "content": summary},
        ] + recent
    
    def get_messages(self) -> List[Dict[str, str]]:
        """Get the current messages for the LLM call."""
        result = [{"role": "system", "content": self.system_prompt}]
        result.extend(self.messages)
        return result
    
    def get_stats(self) -> Dict[str, Any]:
        """Get memory usage statistics."""
        total = self._total_tokens()
        return {
            "total_tokens": total,
            "max_tokens": self.max_tokens,
            "usage_pct": total / self.max_tokens * 100,
            "num_messages": len(self.messages),
            "num_summaries": len(self.summaries),
        }


print("WorkingMemoryManager defined.")

In [ ]:
# Demo: Working memory management
wmm = WorkingMemoryManager(max_tokens=500, summary_threshold=0.7)  # Low limits for demo
wmm.set_system_prompt("You are a helpful assistant with tools.")

# Simulate an agent running many steps
wmm.add_message("user", "Compare RLHF and DPO for alignment.")

steps = [
    ("assistant", "Thought: I need to search for RLHF.\nAction: Search\nAction Input: RLHF"),
    ("user", "Observation: RLHF uses SFT + reward model + PPO. Key paper: InstructGPT."),
    ("assistant", "Thought: Now search for DPO.\nAction: Search\nAction Input: DPO"),
    ("user", "Observation: DPO uses classification loss on preference pairs. Simpler than RLHF."),
    ("assistant", "Thought: Let me compare the math.\nAction: Search\nAction Input: DPO vs RLHF math"),
    ("user", "Observation: RLHF: r(x,y) trained separately, then PPO. DPO: policy directly from preferences."),
    ("assistant", "Thought: Now for practical differences.\nAction: Search\nAction Input: RLHF DPO practical"),
    ("user", "Observation: RLHF: more compute, more flexible. DPO: less compute, less flexible."),
    ("assistant", "Thought: Let me also check recent alternatives.\nAction: Search\nAction Input: KTO IPO ORPO"),
    ("user", "Observation: KTO uses binary feedback. IPO adds regularization. ORPO combines SFT+alignment."),
]

for role, content in steps:
    wmm.add_message(role, content)
    stats = wmm.get_stats()
    print(f"After adding {role} message: {stats['num_messages']} msgs, "
          f"{stats['total_tokens']} tokens ({stats['usage_pct']:.0f}%), "
          f"{stats['num_summaries']} summaries")

print("\n--- Final messages sent to LLM ---")
for msg in wmm.get_messages():
    print(f"  [{msg['role']}]: {msg['content'][:80]}...")

---
## 10. Putting It Together: Planning Agent with Memory

In [ ]:
class PlanningAgent:
    """A complete agent that combines:
    - Task decomposition (planning)
    - ReAct execution (acting)
    - Vector memory (long-term recall)
    - Episodic memory (learning from past tasks)
    - Working memory management (context window)
    
    This is the architecture you'd describe in a system design interview.
    """
    
    def __init__(
        self,
        tools: Optional[List[Tool]] = None,
        verbose: bool = True,
        use_mock: bool = True,
    ):
        self.verbose = verbose
        self.tools = tools or [CalculatorTool(), SearchTool(), PythonREPLTool()]
        
        # Components
        self.decomposer = TaskDecomposer()
        self.executor_agent = SimpleReActAgent(tools=self.tools, verbose=verbose)
        self.plan_executor = PlanExecutor(agent=self.executor_agent, verbose=verbose)
        self.long_term_memory = VectorMemory(use_mock=use_mock)
        self.episodic_memory = EpisodicMemory(use_mock=use_mock)
    
    def run(self, task: str) -> Dict[str, Any]:
        """Run the full planning agent pipeline."""
        start_time = time.time()
        
        if self.verbose:
            print(f"\n{'='*70}")
            print(f"PLANNING AGENT")
            print(f"{'='*70}")
            print(f"Task: {task}")
        
        # Step 1: Check episodic memory for similar past tasks
        if self.verbose:
            print(f"\n--- Phase 1: Recall Past Experience ---")
        
        past_lessons = self.episodic_memory.get_lessons_for_task(task)
        if self.verbose:
            print(past_lessons)
        
        # Step 2: Check long-term memory for relevant information
        if self.verbose:
            print(f"\n--- Phase 2: Retrieve Relevant Knowledge ---")
        
        relevant = self.long_term_memory.retrieve(task, k=3)
        if relevant:
            if self.verbose:
                print(f"Found {len(relevant)} relevant memories:")
                for r in relevant:
                    print(f"  [{r['score']:.3f}] {r['text'][:80]}")
        else:
            if self.verbose:
                print("No relevant long-term memories found.")
        
        # Step 3: Decompose the task
        if self.verbose:
            print(f"\n--- Phase 3: Task Decomposition ---")
        
        subtasks = self.decomposer.decompose(task)
        if self.verbose:
            for st in subtasks:
                opt = " [OPT]" if st.is_optional else ""
                print(f"  {st.id}. {st.description}{opt}")
        
        # Step 4: Execute the plan
        if self.verbose:
            print(f"\n--- Phase 4: Plan Execution ---")
        
        plan_result = self.plan_executor.execute(subtasks, overall_goal=task)
        
        # Step 5: Store results in long-term memory
        for st in subtasks:
            if st.result:
                self.long_term_memory.store(
                    st.result[:500],
                    {"task": task, "subtask": st.description, "step": st.id}
                )
        
        # Step 6: Record episode
        duration_ms = (time.time() - start_time) * 1000
        outcome = "success" if plan_result["completed"] == plan_result["total"] else (
            "partial" if plan_result["completed"] > 0 else "failure"
        )
        
        # Generate lessons learned
        lessons = []
        for st in subtasks:
            if st.status == "failed":
                lessons.append(f"Subtask '{st.description[:40]}' failed — consider alternative approach")
            elif st.status == "skipped":
                lessons.append(f"Subtask '{st.description[:40]}' was skipped (optional)")
        if plan_result["completed"] == plan_result["total"]:
            lessons.append("Plan completed successfully — this decomposition worked well")
        
        self.episodic_memory.record_episode(
            task=task,
            plan=[st.description for st in subtasks],
            outcome=outcome,
            result=str(plan_result["results"]),
            lessons_learned=lessons,
            duration_ms=duration_ms,
        )
        
        # Summary
        if self.verbose:
            print(f"\n{'='*70}")
            print(f"EXECUTION SUMMARY")
            print(f"{'='*70}")
            print(f"Outcome: {outcome}")
            print(f"Subtasks: {plan_result['completed']}/{plan_result['total']} completed")
            print(f"Duration: {duration_ms:.0f}ms")
            print(f"Long-term memories: {len(self.long_term_memory)}")
            print(f"Episodes recorded: {len(self.episodic_memory.episodes)}")
            if lessons:
                print(f"Lessons learned:")
                for l in lessons:
                    print(f"  - {l}")
        
        return {
            "outcome": outcome,
            "plan_result": plan_result,
            "duration_ms": duration_ms,
            "lessons": lessons,
        }


print("PlanningAgent defined.")

In [ ]:
# Run the planning agent on a complex task
planning_agent = PlanningAgent(verbose=True, use_mock=True)

# First task
result1 = planning_agent.run(
    "Write a research summary comparing RLHF and DPO, including their "
    "mathematical foundations, practical trade-offs, and recent developments."
)

In [ ]:
# Second task — the agent should recall lessons from the first task
print("\n" + "*" * 70)
print("SECOND TASK — agent now has episodic memory of first task")
print("*" * 70)

result2 = planning_agent.run(
    "Compare PPO-based RLHF with DPO for aligning language models. "
    "Focus on computational cost and data efficiency."
)

---
## 11. "Why Does This Work?"

### Planning vs Reactive: When is planning worth the overhead?

**Planning is worth it when:**
- The task has 5+ steps
- Steps have dependencies (order matters)
- Failure of one step requires choosing an alternative path
- The task is ambiguous and needs decomposition to clarify

**Planning is NOT worth it when:**
- The task is simple (1-2 steps)
- The decomposition is obvious (just do A then B)
- Speed matters more than quality
- The planning LLM call costs as much as just doing the task

### RAG vs Fine-tuning for Memory?

| Approach | When to use | Pros | Cons |
|----------|-------------|------|------|
| **RAG (retrieval)** | Dynamic knowledge, per-user data | No retraining, updatable | Retrieval quality limits, latency |
| **Fine-tuning** | Stable knowledge, behavioral changes | Fast inference, deep integration | Expensive, stale, catastrophic forgetting |
| **Both** | Best of both worlds | Accuracy + speed | Complexity, cost |

For agent memory, RAG is almost always the right choice because:
1. Agent observations change every run (fine-tuning can't keep up)
2. You need per-session and per-user memory (fine-tuning is global)
3. You need to forget (fine-tuning doesn't support deletion)

### What happens when the plan is wrong?

Plans often fail because:
1. **Decomposition is wrong**: LLM misunderstands the task structure
2. **Subtask fails**: A tool returns an error or irrelevant results
3. **Dependencies are wrong**: Step 3 needs info from step 4
4. **New information changes the plan**: Observation reveals the approach won't work

**Re-planning strategies:**
- **Local retry**: Re-run the failed subtask with modified input
- **Skip optional**: Mark non-critical subtasks as optional, skip on failure
- **Full re-plan**: Feed the failure observation back to the decomposer, generate new plan
- **Escalate**: Ask the user for guidance (human-in-the-loop)

The key insight: **plans should be treated as hypotheses, not commitments**. Be ready to re-plan.

---
## Interview Question Bank

*Planning and memory questions are favorites at Google DeepMind and Anthropic because they connect to classical AI (search, planning) and reveal whether you can think beyond the current prompt-in/text-out paradigm.*

---

### Q1: "Design a memory system for an agent that runs continuously for weeks" -- SYSTEM DESIGN (30 min)

**What this tests**: Can you design for scale and longevity? Most candidates only think about single-session agents.

**Good answer** (hire):
- Vector store (e.g., Pinecone, Weaviate) for long-term semantic memory
- Summarization for working memory (compress old context to stay within the context window)
- Clear separation of short-term (current session) vs long-term (persistent) memory

**Great answer** (strong hire): All of the above, plus:
- **Memory consolidation**: periodically merge similar memories to prevent redundancy (analogous to hippocampal replay in neuroscience)
- **Forgetting strategies**: relevance decay -- memories accessed recently or frequently are weighted higher, stale memories are pruned or archived
- **Hierarchical memory**: session-level (what happened in the last 5 minutes) -> episode-level (what happened today) -> long-term (key facts and lessons learned across weeks)
- **Memory capacity planning**: at 1536-dim embeddings, 1M memories = ~6GB. Plan for this. Use approximate nearest neighbor (HNSW) for sub-100ms retrieval.
- **Contradiction detection**: when new information contradicts existing memory, flag it rather than silently overwriting

**Red flag**: Only mentions "just use a vector database" without discussing consolidation, forgetting, or scale. Does not consider what happens when the memory store grows to millions of entries.

**Follow-up 1**: "How do you handle contradictory memories?"
- Good: timestamp-based (newer wins)
- Great: depends on the domain. For facts, newer usually wins. For preferences, ask the user to clarify. For learned strategies, keep both and let the agent reason about which applies. Mentions the frame problem in classical AI.

**Follow-up 2**: "Your agent's memory database has 1M entries and retrieval is slow. Optimize."
- Good: use HNSW index, increase batch size
- Great: discusses the full optimization stack -- (1) better embeddings reduce the number of retrievals needed, (2) metadata filtering before vector search narrows the candidate set, (3) hierarchical retrieval (coarse search first, then fine), (4) caching frequent queries, (5) quantization (int8 embeddings cut storage 4x with minimal quality loss)

---

### Q2: "When should an agent plan ahead vs react in the moment?"

**What this tests**: Understanding of when planning is worth the overhead. This connects to deep RL concepts (model-based vs model-free).

**Good answer**: Complex tasks with many steps need planning. Simple single-step tasks do not.

**Great answer**: Discusses the specific conditions and trade-offs:

**Plan when**:
- Task has 5+ dependent steps where order matters
- Mistakes are expensive or irreversible (database operations, financial transactions)
- Subtasks can be parallelized (planning reveals the dependency graph)
- You need to allocate a limited budget across steps

**React when**:
- Task is simple (1-3 steps) -- planning overhead exceeds execution time
- Environment is highly dynamic (plans go stale immediately)
- Feedback is immediate and cheap (web browsing -- just try and see)

**The planning horizon problem**: Plans degrade over time. A 20-step plan is almost certainly wrong by step 10. The solution is **re-planning**: plan 5 steps, execute 3, re-plan from the new state. This connects to model-predictive control in robotics and receding horizon planning in RL.

**Red flag**: Gives a binary answer ("always plan" or "never plan"). Does not discuss plan staleness or re-planning.

---

### Q3: "Implement Tree of Thoughts search for a problem-solving agent" -- LIVE CODING (20 min)

**What this tests**: Can you implement a search algorithm over LLM-generated thoughts?

**Good answer**: Working implementation with:
- Branching: generate N candidate next-thoughts from each state
- Evaluation: score each candidate (LLM-as-judge or heuristic)
- Search: BFS or DFS over the thought tree
- Solution extraction: return the best path

**Great answer**: All of the above, plus:
- Pruning: discard low-scoring branches early to save compute
- Configurable branching factor and depth (not hardcoded)
- Discusses BFS vs DFS trade-off: BFS explores broadly (better for problems where the right direction is unclear), DFS explores deeply (better for problems where the right direction is clear but needs refinement)
- Cost awareness: ToT uses O(b^d) LLM calls where b = branching factor, d = depth. For b=3, d=4, that is 81 LLM calls. Expensive.

**Red flag**: Cannot implement basic tree search. Does not consider the cost implications.

---
## Production Implementation Notes

*How memory and planning actually work in deployed agent systems.*

### Memory at Scale

| Component | Technology | When to Use | Cost |
|-----------|-----------|-------------|------|
| **Structured data** | Redis / PostgreSQL | User preferences, task state, tool configs | Low -- pennies per GB |
| **Semantic memory** | Vector DB (Pinecone, Weaviate, Qdrant) | Long-term knowledge, similar experience retrieval | Medium -- $0.10/1M vectors/month |
| **Hybrid retrieval** | Vector + BM25 (keyword) | When semantic alone misses exact matches | Medium -- requires dual index |
| **Working memory** | In-context (prompt) | Current task state, recent observations | High -- every token costs API $ |

### Planning in Production

**Plans are cheap to generate but expensive to execute.** A 10-step plan costs ~$0.05 to generate but each step might cost $0.50-5.00 to execute (API calls, tool use, human review). This means:
- Always validate a plan before executing it (sanity check: are the tools available? are the arguments reasonable?)
- Prefer incremental execution with checkpoints over fire-and-forget execution of all steps
- Log the plan alongside the execution trace so you can diagnose "was the plan bad, or was the execution bad?"

### Context Window Budget Management

LLM prices are quoted per direction (input and output tokens are billed at different rates) and change frequently -- check the current rate cards before quoting numbers in an interview. As one dated example: in early 2025, Claude Opus input tokens were ~$15/1M. A 100-step agent session can easily cost **$5-20**, so budget management is not optional.

**Practical strategies**:
- **Sliding window**: keep only the last N observations in context, summarize older ones
- **Selective retrieval**: do not dump all memory into every prompt -- retrieve only what is relevant to the current step
- **Token counting**: track cumulative tokens per session, alert or terminate when budget is exceeded
- **Tiered models**: use Opus/GPT-4o for planning and difficult steps, Haiku/GPT-4o-mini for simple tool calls

### The MemGPT Pattern (Production Memory Management)

The most important production pattern for long-running agents:
1. **Main context** (limited): current task, recent history, retrieved memories
2. **Archival storage** (unlimited): all past interactions, searchable via embedding
3. **Memory manager**: an LLM-controlled process that decides what to move from main context to archival (and what to retrieve back)

This is a MemGPT-style design; vendor implementations (e.g., Claude's memory, ChatGPT's memory) are not public, so treat this as the pattern they popularized rather than how those products work under the hood.

---
## How This Gets Tested in Interviews

### The Planning + Memory Interview Pattern

Planning and memory questions almost always come as **system design** questions, not coding questions. The interviewer describes a scenario ("design an agent that...") and evaluates your ability to decompose the problem.

### What Interviewers Listen For

**When you discuss planning**:
- Do you mention re-planning? (Most candidates describe static plans and never revisit them)
- Do you discuss the cost of planning vs the cost of failure? (Planning a 3-step task is wasteful; planning a 30-step irreversible task is essential)
- Can you connect to classical AI? ("This is essentially hierarchical task network planning" or "this relates to MCTS" -- shows intellectual depth)

**When you discuss memory**:
- Do you distinguish types of memory? (Semantic, episodic, procedural -- not just "vector database")
- Do you mention scale? ("At 10K memories this is fine, at 1M we need HNSW indexing and possibly sharding")
- Do you think about what to forget? Most candidates only discuss what to remember. Forgetting is equally important and much harder.

### Common Pitfalls

1. **Over-engineering the plan**: Candidate designs a 5-component planning system for a task that needs 3 steps. Shows lack of judgment about when planning is worth the overhead.

2. **Ignoring memory limits**: "Just put everything in the context window." At 200K tokens, that is $3 per query. Not viable for production.

3. **No concrete numbers**: When asked "how would you store 1M memories?", give actual numbers. "1536-dim float32 embeddings = 6GB. HNSW index adds 2x overhead. Total ~18GB. Fits on a single machine. Retrieval latency: ~10ms for top-10 with HNSW." Concrete numbers demonstrate real experience.

### The "Weeks" Question Is a Trap

When an interviewer asks about an agent running for weeks, they are really testing whether you understand:
- **Statefulness**: Where does state live between sessions? (Not in RAM.)
- **Drift**: The agent's behavior may drift over time as memory accumulates and the world changes
- **Recovery**: What happens when the agent crashes? Can it resume?
- **Cost**: A continuously running agent at $0.01/request and 1 request/minute = $14.40/day = $100/week. You need to justify this cost.

---
## 12. Flashcard Summary

Study these Q&A pairs for interview preparation.

| # | Question | Answer |
|---|----------|--------|
| 1 | What are the 3 types of agent memory? | **Short-term** (context window), **Long-term** (vector store), **Episodic** (past task experiences) |
| 2 | What is task decomposition? | Breaking a complex goal into ordered subtasks with dependencies, each small enough for a ReAct agent to execute |
| 3 | What is Tree of Thoughts? | Exploring multiple reasoning paths (branching), evaluating each, and backtracking when needed. Uses LLM as both generator and evaluator |
| 4 | How does ToT differ from CoT? | CoT: single linear path. ToT: tree of paths with evaluation, pruning, and backtracking |
| 5 | When should you use ToT over CoT? | When problems have multiple valid approaches, when backtracking is needed, when you can evaluate intermediate progress |
| 6 | What is episodic memory for agents? | Storing past task attempts with outcomes and lessons learned. Enables "I tried X before and it failed because Y" |
| 7 | How does a vector store enable long-term memory? | Embed text into vectors, store in FAISS/Pinecone. Retrieve by semantic similarity. Agent can recall relevant past observations |
| 8 | What is working memory management? | Summarizing old context to stay within token limits. Keep recent observations in full, compress old ones |
| 9 | When does planning fail? | Wrong decomposition, subtask failures, wrong dependencies, new information invalidates the plan |
| 10 | What is re-planning? | Generating a new plan when the current one fails. Feed failure observations back to the decomposer |
| 11 | RAG vs fine-tuning for agent memory? | RAG: dynamic, updatable, per-session. Fine-tuning: static, global, expensive. For agents, RAG is almost always correct |
| 12 | What is the context window problem for agents? | Each step adds tokens. After many steps, old observations are pushed out. Solution: working memory management with summarization |
| 13 | How do you evaluate a planning agent? | Task completion rate, step efficiency, re-plan frequency, memory utilization, latency |
| 14 | What is BFS vs DFS in Tree of Thoughts? | BFS: explore all candidates at each depth, keep top k. DFS: go deep on best candidate first, backtrack if needed |
| 15 | What is the biggest practical challenge for agents with memory? | Memory quality — garbage-in-garbage-out. If you store bad observations, retrieval returns bad context, which leads to worse decisions |

---
## 13. Paper Guides

### Paper 1: Toolformer (Schick et al. 2023)

**Paper**: [Toolformer: Language Models Can Teach Themselves to Use Tools](https://arxiv.org/abs/2302.04761)  
**Venue**: NeurIPS 2023  
**Reading time**: ~40 minutes  

**Section-by-Section Guide**

**Abstract & Introduction (5 min)**
- Key claim: LLMs can learn to use tools through self-supervised learning
- No need for human annotation of when to use tools
- Tools: calculator, calendar, search engine, translation, Q&A system

**Section 2: Approach (15 min)** — THE KEY SECTION
- Step 1: Sample API calls — the model generates candidate positions to insert tool calls
- Step 2: Execute them — run the tools and get results
- Step 3: Filter — keep only calls that improve perplexity (the text is more likely with the tool result than without)
- Step 4: Fine-tune — train the model on the filtered dataset
- Key insight: the model learns WHEN tools are useful, not just HOW to call them

**Section 3: Experiments (10 min)**
- Toolformer outperforms GPT-3 (175B) on many tasks with only 6.7B parameters + tools
- The perplexity filter is crucial — without it, the model calls tools unnecessarily
- Table 1 and 2: strong results on math, QA, temporal reasoning

**Key Takeaway for Interviews**
> "Toolformer shows that LLMs can self-supervise their tool usage. By filtering for calls that reduce perplexity, the model learns not just how to use tools, but when they're actually helpful. This eliminates the need for human annotation of tool-use decisions."

**Limitations**: Only text-in/text-out tools. No multi-step tool chains. Training required (can't just prompt).

---

### Paper 2: Tree of Thoughts (Yao et al. 2023)

**Paper**: [Tree of Thoughts: Deliberate Problem Solving with Large Language Models](https://arxiv.org/abs/2305.10601)  
**Venue**: NeurIPS 2023  
**Reading time**: ~35 minutes  

**Section-by-Section Guide**

**Abstract & Introduction (5 min)**
- Extends chain-of-thought from a single chain to a tree
- LLM serves as both thought generator and evaluator
- Enables deliberate search: BFS, DFS, or best-first

**Section 2: Background (5 min)**
- IO prompting → CoT prompting → Self-Consistency → Tree of Thoughts
- Each is a generalization of the previous

**Section 3: ToT Framework (10 min)** — KEY SECTION
- 4 questions to define a ToT:
  1. How to decompose into thought steps?
  2. How to generate candidate thoughts?
  3. How to evaluate states?
  4. What search algorithm to use?
- Figure 1: the visual comparison (crucial for understanding)

**Section 4: Experiments (10 min)**
- Game of 24: IO prompting scores 7.3%, CoT only 4.0%, ToT (b=5) reaches 74% success rate
- Creative writing: ToT produces more coherent stories
- Mini crosswords: ToT solves via backtracking
- Key: these are problems where the first approach often fails and backtracking helps

**Key Takeaway for Interviews**
> "Tree of Thoughts extends chain-of-thought by treating reasoning as search — generating multiple candidate thoughts, evaluating them, and backtracking when needed. It's most useful for problems where the first approach often fails, like puzzle solving or creative tasks. The trade-off is cost: ToT uses 10-100x more LLM calls than CoT."

**Limitations**: Very expensive (many LLM calls per problem). Evaluation quality depends on the LLM. Doesn't scale well to long chains (diminishing returns).

### Additional Paper: MemGPT / Letta (Packer et al., 2023)

**Paper**: [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560)

**Key Idea**: Manages LLM memory like an OS manages virtual memory. The LLM has a limited "main context" (like RAM) and can page information in/out from "external memory" (like disk). The system provides function calls for memory management: `core_memory_append`, `core_memory_replace`, `archival_memory_insert`, `archival_memory_search`.

**Why It Matters for Agents**:
- Solves the context window limitation for long-running agents
- The agent itself decides what to remember and what to forget
- Creates the illusion of infinite memory within a finite context window
- Now productionized as [Letta](https://www.letta.com/) -- an open-source framework for building stateful agents

**Interview Relevance**: If asked "how would you build an agent that runs for hours/days?", MemGPT's virtual memory approach is the key reference. It shows you understand that context window management is a first-class architectural concern, not an afterthought.

In [ ]:
print("Notebook 14 complete!")
print("\nKey takeaways:")
print("1. Planning decomposes complex goals into manageable subtasks")
print("2. Tree of Thoughts explores multiple paths with evaluation and backtracking")
print("3. Three memory types: short-term (context), long-term (vectors), episodic (experiences)")
print("4. Working memory management prevents context window overflow")
print("5. Plans are hypotheses — always be ready to re-plan")
print("6. Episodic memory enables agents to learn from past successes and failures")
print("\nFor interviews: be ready to whiteboard the PlanningAgent architecture")
print("(decomposer -> executor -> memory -> re-planner)")